# InfoDengue - Brazilian Arbovirus Surveillance

This notebook demonstrates how to access epidemiological surveillance data for **arbovirus diseases** (Dengue, Chikungunya, and Zika) in Brazil using the **InfoDengue** accessor.

**Data Source:** [InfoDengue project](https://info.dengue.mat.br/) via the [Mosqlimate API](https://api.mosqlimate.org/)

**Coverage:**
- **Geographic:** All 27 Brazilian states (Federative Units)
- **Temporal:** Weekly case notifications
- **Diseases:** Dengue, Chikungunya, Zika

> **Note:** Live data retrieval requires a **Mosqlimate API key**.  
> Get one at https://api.mosqlimate.org/ and set it as an environment variable:
> ```bash
> export MOSQLIMATE_API_KEY="your-key-here"
> ```

**Requirements:**
```bash
pip install pandas matplotlib seaborn epidatasets
```

## 1. Setup and Imports

In [1]:
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
%matplotlib inline

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)

print(f"Imports completed. {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Imports completed. 2026-06-22 10:00


## 2. Initialize the InfoDengue Accessor

The `InfoDengueAPI` accessor can be created through the unified registry with `get_source("infodengue")`, or instantiated directly.

It will look for the API key in:
1. The `MOSQLIMATE_API_KEY` environment variable
2. Config files (`~/.nanobot/config/mosqlimate.env`, `~/.config/epi_data/mosqlimate.env`, `./.env`)
3. The `api_key` constructor argument

In [2]:
# Register a dummy API key so we can explore the accessor's metadata methods.
# Replace with your real key (or set MOSQLIMATE_API_KEY) to fetch live data.
api_key = os.getenv('MOSQLIMATE_API_KEY', 'demo-key')

from epidatasets.sources.infodengue_api import InfoDengueAPI

api = InfoDengueAPI(api_key=api_key)

print(f"Source      : {api.source_name}")
print(f"Description : {api.source_description}")
print(f"API base    : {api.BASE_URL}")

Source      : infodengue
Description : Epidemiological data from the InfoDengue project, which monitors dengue, chikungunya, and Zika cases in Brazil via the Mosqlimate API.
API base    : https://api.mosqlimate.org/api


## 3. Discover Available Data

These metadata methods work **without** making live API requests, so they are useful for discovery and validation.

In [3]:
# Supported diseases
diseases = api.get_diseases()
print("Diseases monitored by InfoDengue:")
display(diseases)

Diseases monitored by InfoDengue:
   code        name
0  dengue      Dengue
1  chikungunya  Chikungunya
2  zika        Zika


,code,name
0,dengue,Dengue
1,chikungunya,Chikungunya
2,zika,Zika


In [4]:
# Brazilian states (Federative Units)
states = api.get_states()
print(f"Brazilian states covered ({len(states)} total):")
for _, row in states.head(5).iterrows():
    print(f"  {row['code']} -> {row['name']}")
print("  ...")
for _, row in states.tail(3).iterrows():
    print(f"  {row['code']} -> {row['name']}")

Brazilian states covered (27 total):
  AC -> Acre
  AL -> Alagoas
  AP -> Amapá
  AM -> Amazonas
  BA -> Bahia
  ...
  SP -> São Paulo
  SE -> Sergipe
  TO -> Tocantins


In [5]:
# Country coverage (InfoDengue is Brazil-only)
countries = api.list_countries()
print("InfoDengue country coverage:")
print(countries.to_string(index=False))

InfoDengue country coverage:
  country_code country_name
0           BR       Brazil


## 4. Fetching Case Data

The `get_cases()` method retrieves weekly case notifications. You can filter by:
- **disease**: `dengue`, `chikungunya`, or `zika`
- **start_date / end_date**: date range (`YYYY-MM-DD`)
- **uf**: Brazilian state abbreviation (e.g., `SP`, `RJ`)
- **geocode**: IBGE municipality code

Responses are cached for 24 hours by default to reduce API load.

In [6]:
# Example: fetch Dengue cases for São Paulo state in 2024.
# This call hits the live Mosqlimate API, so it requires a real API key.
if os.getenv('MOSQLIMATE_API_KEY'):
    dengue_sp = api.get_cases(
        disease='dengue',
        start_date='2024-01-01',
        end_date='2024-12-31',
        uf='SP',
    )
    print(f"Fetched {len(dengue_sp)} records")
    display(dengue_sp.head())
else:
    print("⚠ Live API call skipped (no valid MOSQLIMATE_API_KEY set).")
    print("To fetch real data, set MOSQLIMATE_API_KEY and re-run this cell.")

⚠ Live API call skipped (no valid MOSQLIMATE_API_KEY set).
To fetch real data, set MOSQLIMATE_API_KEY and re-run this cell.


### Fetch a full year with pagination

The convenience method `get_cases_brazil()` automatically paginates through all results for a given disease and year.

In [7]:
# Fetch all dengue cases for Brazil in 2024 (auto-paginates)
if os.getenv('MOSQLIMATE_API_KEY'):
    dengue_brazil_2024 = api.get_cases_brazil(disease='dengue', year=2024)
    print(f"Total dengue records in 2024: {len(dengue_brazil_2024)}")
    display(dengue_brazil_2024.head())
else:
    print("⚠ Live API call skipped (no valid MOSQLIMATE_API_KEY set).")

⚠ Live API call skipped (no valid MOSQLIMATE_API_KEY set).


### Convenience function

For quick one-off queries, use the module-level `get_dengue_cases()` helper.

In [8]:
from epidatasets.sources.infodengue_api import get_dengue_cases

# Equivalent to: api.get_cases_brazil(disease='dengue', year=2023) filtered to RJ
if os.getenv('MOSQLIMATE_API_KEY'):
    rj_dengue = get_dengue_cases(uf='RJ', year=2023)
    print(f"Rio de Janeiro dengue cases (2023): {len(rj_dengue)} records")
else:
    print("ℹ Set MOSQLIMATE_API_KEY to run get_dengue_cases(uf='RJ', year=2023)")

ℹ Set MOSQLIMATE_API_KEY to run get_dengue_cases(uf='RJ', year=2023)


## 5. Visualizing Case Data

Below is a reusable visualization template. Once you have fetched a DataFrame with a date column and case counts, you can plot the weekly time series.

In [9]:
# Template: plot a weekly time series of dengue cases.
# Replace `df` with the output of api.get_cases() when a real API key is set.
import numpy as np

# Simulated weekly series for demonstration purposes only.
weeks = pd.date_range('2024-01-07', periods=52, freq='W')
np.random.seed(42)
cases = np.abs(np.random.poisson(lam=300, size=52) * (1 + np.sin(np.arange(52) / 4)))
df = pd.DataFrame({'week': weeks, 'cases': cases.astype(int)})

fig, ax = plt.subplots()
ax.plot(df['week'], df['cases'], marker='o', linewidth=2)
ax.set_title('Simulated Weekly Dengue Cases (template)', fontsize=14)
ax.set_xlabel('Epidemiological week')
ax.set_ylabel('Reported cases')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Loaded simulated template data with {len(df)} weekly points.")

Loaded simulated template data with 52 weekly points.


## 6. Compare Diseases

InfoDengue tracks three arboviruses. Fetch each disease separately and compare their seasonal patterns.

In [10]:
# Template for a multi-disease comparison.
# With a valid API key, fetch each disease for the same period/state and plot together.
if os.getenv('MOSQLIMATE_API_KEY'):
    disease_dfs = {}
    for d in api.DISEASES:
        disease_dfs[d] = api.get_cases(disease=d, uf='MG', start_date='2024-01-01', end_date='2024-12-31')
    print({k: len(v) for k, v in disease_dfs.items()})
else:
    print("To compare diseases, fetch each with api.get_cases(disease=d) for d in", list(api.DISEASES))


To compare diseases, fetch each with api.get_cases(disease=d) for d in ['dengue', 'chikungunya', 'zika']


## 7. Cache Management

The accessor caches API responses to disk (default: `~/.cache/epi_data/infodengue/`) with a configurable TTL. You can clear the cache to force fresh downloads.

In [11]:
print(f"Cache directory: {api.cache_dir}")
print(f"Cache TTL: {api._cache_ttl}")

# Remove all cached responses
api.clear_cache()
print("Cache cleared.")

Cache directory: /root/.cache/epi_data/infodengue
Cache TTL: 1 day, 0:00:00
Cache cleared.


## 8. Summary

| Method | Description |
|--------|-------------|
| `get_cases(disease, start_date, end_date, uf, geocode)` | Weekly case data with filters |
| `get_cases_brazil(disease, year)` | Full-year data with auto-pagination |
| `get_states()` | List of 27 Brazilian states |
| `get_diseases()` | Supported diseases (dengue, chikungunya, zika) |
| `list_countries()` | Country coverage (Brazil) |
| `clear_cache()` | Clear the on-disk response cache |

**Next steps:**
- Combine InfoDengue data with climate variables from the [Copernicus CDS](21_Copernicus_CDS_Examples.ipynb) accessor to study environmental drivers of dengue transmission.
- Cross-reference with genomic surveillance via [Pathoplexus](19_Pathoplexus_Genomic_Data.ipynb).
- Pair with [DATASUS / PySUS](01_pysus_brazilian_health_data.ipynb) for hospitalisation and mortality context.